# 05 Modelo Segmentacion Clientes

Este notebook entrena el tercer modelo del proyecto usando **K-Means** para segmentar clientes segun su comportamiento historico de compra.

## 1. Librerias

Se importan las librerias necesarias para la construccion de la base agregada y el clustering.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 2. Cargar la base EDA

Se usa el parquet de la etapa 02 para generar una base agregada por cliente.

In [ ]:
# Definir la ruta base del proyecto.
PROJECT_ROOT = Path.cwd().resolve().parents[1]

# Apuntar al parquet del EDA.
PARQUET_PATH = PROJECT_ROOT / "parquets" / "02_EDA_Base_Tickets" / "02_base_eda_tickets.parquet"

# Cargar la base para segmentacion.
df = pd.read_parquet(PARQUET_PATH)

# Convertir fecha para calculos temporales.
df["fecha"] = pd.to_datetime(df["fecha"])
df.head()

## 3. Construir la base por cliente

Se agregan los tickets para obtener indicadores historicos de cada cliente.

In [ ]:
# Obtener la fecha maxima para calcular recencia.
fecha_max = df["fecha"].max()

# Construir la base agregada a nivel cliente.
clientes = (
    df.groupby("id_cliente", dropna=False)
    .agg(
        numero_tickets=("id_ticket_modelado", "nunique"),
        gasto_total=("total_pedido", "sum"),
        ticket_promedio=("total_pedido", "mean"),
        ticket_maximo=("total_pedido", "max"),
        cantidad_total_productos=("cantidad_total", "sum"),
        promedio_productos_ticket=("cantidad_total", "mean"),
        categorias_distintas_totales=("categorias_distintas", "sum"),
        ultimo_ticket=("fecha", "max"),
        primer_ticket=("fecha", "min"),
        metodo_pago_frecuente=("metodo_pago", lambda s: s.mode().iloc[0] if not s.mode().empty else "Sin dato"),
    )
    .reset_index()
)

# Derivar columnas de permanencia y recencia.
clientes["dias_activos"] = (clientes["ultimo_ticket"] - clientes["primer_ticket"]).dt.days + 1
clientes["dias_desde_ultimo_ticket"] = (fecha_max - clientes["ultimo_ticket"]).dt.days

# Mostrar las primeras filas de la base por cliente.
clientes.head()

## 4. Seleccionar variables de clustering

Aqui se define el conjunto de variables que alimentan a K-Means.

In [ ]:
# Definir las variables numericas para segmentacion.
FEATURES = [
    "numero_tickets",
    "gasto_total",
    "ticket_promedio",
    "ticket_maximo",
    "cantidad_total_productos",
    "promedio_productos_ticket",
    "categorias_distintas_totales",
    "dias_activos",
    "dias_desde_ultimo_ticket",
]

# Revisar la matriz base de clustering.
clientes[FEATURES].head()

## 5. Entrenar el modelo

Se escala la base y luego se entrena el modelo K-Means con tres clusters.

In [ ]:
# Definir el preprocesador para escalar variables numericas.
preprocessor = ColumnTransformer(
    transformers=[("num", Pipeline(steps=[("scaler", StandardScaler())]), FEATURES)],
    remainder="drop",
)

# Definir el pipeline completo de clustering.
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("clusterer", KMeans(n_clusters=3, random_state=42, n_init=20)),
])

# Entrenar el modelo con la base agregada.
pipeline.fit(clientes[FEATURES])

# Asignar cluster a cada cliente.
clientes["cluster"] = pipeline.predict(clientes[FEATURES])
clientes.head()

## 6. Evaluacion de la segmentacion

Se calculan metricas simples para validar la calidad general de los clusters.

In [ ]:
# Transformar la base para calcular silhouette.
X_transformed = pipeline.named_steps["preprocessor"].transform(clientes[FEATURES])

# Calcular la metrica silhouette y la inercia.
metricas = {
    "n_clientes": int(len(clientes)),
    "n_clusters": 3,
    "inercia": round(float(pipeline.named_steps["clusterer"].inertia_), 4),
    "silhouette": round(float(silhouette_score(X_transformed, clientes["cluster"])), 4),
}

pd.DataFrame(metricas.items(), columns=["metrica", "valor"])

## 7. Revisar tamano de clusters

Se valida cu?ntos clientes quedaron en cada grupo.

In [ ]:
# Contar clientes por cluster.
clientes["cluster"].value_counts().sort_index()